# 07 - Health Index

Compute the five health components (response time, merge rate, diversity,
trend, bus factor), combine them into a weighted composite health index,
rank repos, and visualise top and bottom performers.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from oss_pulse.analyze.health_index import (
    DEFAULT_WEIGHTS,
    compute_health_components,
    compute_health_index,
    rank_repos,
)
from oss_pulse.visualize.style import PALETTE, setup_style

setup_style()

In [ ]:
# Load monthly aggregated data
DATA_DIR = Path("../data/processed")
monthly_df = pd.read_parquet(DATA_DIR / "repo_monthly.parquet")

print(f"Monthly data: {monthly_df.shape}")
print(f"Repos: {monthly_df['repo_name'].nunique()}")
print(f"\nDefault weights: {DEFAULT_WEIGHTS}")

In [ ]:
# Compute health components (normalised 0-100)
# TODO: run with real data
components = compute_health_components(monthly_df)
print("Health components (normalised 0-100):")
components.round(2)

In [ ]:
# Compute composite health index and rank
# TODO: run with real data
components["health_index"] = compute_health_index(components)
ranked = rank_repos(components)

print("Top 10 repos by health index:")
ranked[["rank", "repo_name", "health_index"]].head(10)

In [ ]:
# Horizontal bar chart: top 10 and bottom 5
# TODO: run with real data
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top 10
top10 = ranked.head(10).sort_values("health_index")
axes[0].barh(
    top10["repo_name"], top10["health_index"],
    color=PALETTE["success"], edgecolor=PALETTE["bg"],
)
axes[0].set_title("Top 10 Healthiest Repos", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Health Index")

# Bottom 5
bottom5 = ranked.tail(5).sort_values("health_index", ascending=False)
axes[1].barh(
    bottom5["repo_name"], bottom5["health_index"],
    color=PALETTE["accent"], edgecolor=PALETTE["bg"],
)
axes[1].set_title("Bottom 5 Repos", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Health Index")

plt.tight_layout()
plt.show()

In [ ]:
# Radar-style component breakdown for top 3 repos
import numpy as np

score_cols = [
    "response_time_score", "merge_rate_score",
    "diversity_score", "trend_score", "bus_factor_score",
]
labels = ["Response\nTime", "Merge\nRate", "Diversity", "Trend", "Bus\nFactor"]

top3 = ranked.head(3)
angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()
angles += angles[:1]  # close the polygon

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={"polar": True})
colors = [PALETTE["primary"], PALETTE["success"], PALETTE["warning"]]

for i, (_, row) in enumerate(top3.iterrows()):
    values = [row[col] for col in score_cols]
    values += values[:1]
    ax.plot(angles, values, color=colors[i], linewidth=1.5, label=row["repo_name"])
    ax.fill(angles, values, color=colors[i], alpha=0.1)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, fontsize=10)
ax.set_title("Health Components: Top 3 Repos", fontsize=14, fontweight="bold", y=1.08)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.show()